# Try Apache Beam - Python

In this notebook, we set up your development environment and work through a simple example using the [DirectRunner](https://beam.apache.org/documentation/runners/direct/). You can explore other runners with the [Beam Capatibility Matrix](https://beam.apache.org/documentation/runners/capability-matrix/).

To navigate through different sections, use the table of contents. From **View**  drop-down list, select **Table of contents**.

To run a code cell, you can click the **Run cell** button at the top left of the cell, or by select it and press **`Shift+Enter`**. Try modifying a code cell and re-running it to see what happens.

To learn more about Colab, see [Welcome to Colaboratory!](https://colab.sandbox.google.com/notebooks/welcome.ipynb).

# Setup

First, you need to set up your environment, which includes installing `apache-beam` and downloading a text file from Cloud Storage to your local file system. We are using this file to test your pipeline.

In [3]:
# Run and print a shell command.
def run(cmd):
  print('>> {}'.format(cmd))
  !{cmd}
  print('')

# Install apache-beam.
# run('pip install --quiet apache-beam')

# Copy the input file into the local file system.
run('mkdir -p data')
run('gsutil cp gs://dataflow-samples/shakespeare/kinglear.txt data/')

>> mkdir -p data

>> gsutil cp gs://dataflow-samples/shakespeare/kinglear.txt data/
Copying gs://dataflow-samples/shakespeare/kinglear.txt...
/ [1 files][153.6 KiB/153.6 KiB]                                                
Operation completed over 1 objects/153.6 KiB.                                    



# Word count with comments

Below is mostly the same code as above, but with comments explaining every line in more detail.

In [5]:
import apache_beam as beam
import re

inputs_pattern = 'data/*'
outputs_prefix = 'outputs/part'

# Running locally in the DirectRunner.
with beam.Pipeline() as pipeline:
  # Store the word counts in a PCollection.
  # Each element is a tuple of (word, count) of types (str, int).
  word_counts = (
      # The input PCollection is an empty pipeline.
      pipeline

      # Read lines from a text file.
      | 'Read lines' >> beam.io.ReadFromText(inputs_pattern)
      # Element type: str - text line

      # Use a regular expression to iterate over all words in the line.
      # FlatMap will yield an element for every element in an iterable.
      | 'Find words' >> beam.FlatMap(lambda line: re.findall(r"[a-zA-Z']+", line))
      # Element type: str - word

      # Create key-value pairs where the value is 1, this way we can group by
      # the same word while adding those 1s and get the counts for every word.
      | 'Pair words with 1' >> beam.Map(lambda word: (word, 1))
      # Element type: (str, int) - key: word, value: 1

      # Group by key while combining the value using the sum() function.
      | 'Group and sum' >> beam.CombinePerKey(sum)
      # Element type: (str, int) - key: word, value: counts
  )

  # We can process a PCollection through other pipelines too.
  (
      # The input PCollection is the word_counts created from the previous step.
      word_counts

      # Format the results into a string so we can write them to a file.
      | 'Format results' >> beam.Map(lambda word_count: str(word_count))
      # Element type: str - text line

      # Finally, write the results to a file.
      | 'Write results' >> beam.io.WriteToText(outputs_prefix)
  )

# Sample the first 20 results, remember there are no ordering guarantees.
run('head -n 200 {}-00000-of-*'.format(outputs_prefix))

>> head -n 200 outputs/part-00000-of-*
('KING', 243)
('LEAR', 236)
('DRAMATIS', 1)
('PERSONAE', 1)
('king', 65)
('of', 447)
('Britain', 2)
('OF', 15)
('FRANCE', 10)
('DUKE', 3)
('BURGUNDY', 8)
('CORNWALL', 63)
('ALBANY', 67)
('EARL', 2)
('KENT', 156)
('GLOUCESTER', 141)
('EDGAR', 126)
('son', 29)
('to', 438)
('Gloucester', 26)
('EDMUND', 99)
('bastard', 7)
('CURAN', 6)
('a', 366)
('courtier', 1)
('Old', 13)
('Man', 11)
('tenant', 3)
('Doctor', 12)
('Fool', 73)
('OSWALD', 53)
('steward', 2)
('Goneril', 12)
('A', 51)
('Captain', 12)
('employed', 1)
('by', 69)
('Edmund', 32)
('Gentleman', 48)
('attendant', 1)
('on', 93)
('Cordelia', 22)
('Herald', 6)
('Servants', 9)
('Cornwall', 12)
('First', 7)
('Servant', 11)
('Second', 4)
('Third', 4)
('GONERIL', 71)
('REGAN', 86)
('daughters', 24)
('Lear', 17)
('CORDELIA', 42)
('Knights', 2)
("Lear's", 4)
('train', 9)
('Captains', 1)
('Messengers', 1)
('Soldiers', 7)
('and', 594)
('Attendants', 8)
('Knight', 8)
('Messenger', 10)
('SCENE', 27)
('ACT', 

---
# Part 2: Improving the lab

Everything above is the course's own lab, unchanged: a word count of *King Lear*. Everything below is new.

1. **What is wrong with the lab's word count**, shown with numbers from its own output.
2. **A fixed word count**: lower case words, a sorted CSV file, `top_n`, stopwords, an input check and run counters.
3. **A second dataset**: eight plays from the same bucket the lab reads.
4. **A comparison of the plays and of two genres**: a custom combiner, a side input and grouping by key.
5. **A dashboard**: a self-contained HTML page with charts.

Run all the cells from top to bottom. In Colab, use *Runtime, Run all*. Each part writes its files to `outputs/`.

In [3]:
# Packages. Inside a notebook, Beam also needs pandas, in the range Beam supports: 1.4.3 up to 2.4.
# Colab usually has pandas already, and may lack apache-beam.
import importlib.metadata as metadata


def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


needs = []
if installed('apache-beam') is None:
    needs.append('apache-beam')
pandas_version = installed('pandas')
if pandas_version is None or tuple(int(part) for part in pandas_version.split('.')[:2]) >= (2, 4):
    needs.append('pandas>=1.4.3,<2.4')
if needs:
    get_ipython().run_line_magic('pip', 'install --quiet ' + ' '.join('"{}"'.format(need) for need in needs))

## 1. What is wrong with the lab's word count

The cell below reads the lab's own output file and checks three things.

- **Case:** `The` and `the` are counted as different words.
- **Quote marks:** a lone `'` is counted as a word.
- **The output** is Python's text form of a tuple, such as `('KING', 243)`, in no order.

There is also a mismatch inside the lab: the comment says "Sample the first 20 results", and the command is `head -n 200`.

In [4]:
import ast
import collections

original = {}
for line in open('outputs/part-00000-of-00001'):
    word, count = ast.literal_eval(line.strip())
    original[word] = count

lowered = collections.Counter()
for word, count in original.items():
    lowered[word.lower()] += count

print('different "words" in the lab output:', len(original))
print('the:', original['the'], '   The:', original['The'])
print('a lone quote mark was counted', original["'"], 'times')
print('different words after lowercasing:', len(lowered))

different "words" in the lab output: 4784
the: 786    The: 122
a lone quote mark was counted 15 times
different words after lowercasing: 4215


## 2. A fixed word count

The pipeline is the lab's, with these changes. Each step keeps its element type in a comment.

- Words are lower case, and a word is letters with single apostrophes inside: `don't` stays one word, and `'tis` counts as `tis`.
- The output is a CSV file with a header, sorted by count and then by word.
- `top_n` keeps the most common words, and an optional stopword file drops very common ones.
- The input is checked first, and the run counts lines, lines without words, and words.

In [5]:
import glob
import os
import re

import apache_beam as beam
from apache_beam.metrics import Metrics, MetricsFilter

In [6]:
def find_words(line):
    """Return every word in one line of text, in lower case.

    A word is letters, with single apostrophes allowed inside it: don't stays one word,
    'tis counts as tis, and a lone quote mark is not a word.
    """
    return re.findall(r"[a-z]+(?:'[a-z]+)*", line.lower())

In [7]:
class FindWords(beam.DoFn):
    """Split each line into words, and count lines, lines without words, and words as it goes."""

    def __init__(self):
        self.lines = Metrics.counter('wordcount', 'lines')
        self.lines_without_words = Metrics.counter('wordcount', 'lines_without_words')
        self.words = Metrics.counter('wordcount', 'words')

    def process(self, line):
        words = find_words(line)
        self.lines.inc()
        if not words:
            self.lines_without_words.inc()
        self.words.inc(len(words))
        yield from words


def load_stopwords(path):
    """Read a stopword file: one word per line, blank lines ignored, case ignored."""
    with open(path) as handle:
        return {line.strip().lower() for line in handle if line.strip()}

In [8]:
def check_input(inputs_pattern):
    """Stop early, with a clear message, if the pattern matches no file or only empty files."""
    paths = glob.glob(inputs_pattern)
    if not paths:
        raise ValueError("no file matches '{}'".format(inputs_pattern))
    if all(os.path.getsize(path) == 0 for path in paths):
        raise ValueError("every file matching '{}' is empty".format(inputs_pattern))


def rank(word_counts, top_n=0):
    """Sort (word, count) pairs by count, highest first, then by word. top_n 0 keeps them all."""
    ranked = sorted(word_counts, key=lambda pair: (-pair[1], pair[0]))
    return ranked[:top_n] if top_n else ranked


def to_csv(ranked):
    """Make the CSV body, one 'word,count' line per pair. Words hold only letters and apostrophes."""
    return '\n'.join('{},{}'.format(word, count) for word, count in ranked)

In [9]:
def build_wordcount_pipeline(pipeline, inputs_pattern, outputs_prefix, top_n=0, stopwords=frozenset()):
    """Attach the word-count steps to a pipeline. Element types are noted after each step."""
    word_counts = (
        pipeline
        | 'Read lines' >> beam.io.ReadFromText(inputs_pattern)  # str, one line
        | 'Find words' >> beam.ParDo(FindWords())  # str, one word
        | 'Drop stopwords' >> beam.Filter(lambda word: word not in stopwords)  # str, one word
        | 'Pair words with 1' >> beam.Map(lambda word: (word, 1))  # (str, int)
        | 'Group and sum' >> beam.CombinePerKey(sum)  # (str, int): word, count
    )
    (
        word_counts
        # Beam gives no order guarantee, so gather the pairs into one list and sort that list.
        # This puts every word on one worker, which is fine for one play and not for a large corpus.
        | 'Gather pairs' >> beam.combiners.ToList()  # one list of (str, int)
        | 'Sort by count' >> beam.Map(rank, top_n)  # one sorted list, cut to top_n
        # One element, so the lines cannot be reordered on the way to the file.
        | 'Make CSV body' >> beam.Map(to_csv)  # one str
        | 'Write results' >> beam.io.WriteToText(
            outputs_prefix, file_name_suffix='.csv', shard_name_template='', header='word,count')
    )

In [10]:
def run_wordcount(inputs_pattern='data/*', outputs_prefix='outputs/word_counts', top_n=20, stopwords_path=None):
    """Run the fixed word count and return its counters. Raises ValueError (or OSError) on bad input."""
    if top_n < 0:
        raise ValueError('top_n must be 0 or more')
    check_input(inputs_pattern)
    stopwords = load_stopwords(stopwords_path) if stopwords_path else frozenset()
    pipeline = beam.Pipeline()
    build_wordcount_pipeline(pipeline, inputs_pattern, outputs_prefix, top_n, stopwords)
    result = pipeline.run()
    result.wait_until_finish()
    counters = result.metrics().query(MetricsFilter().with_namespace('wordcount'))['counters']
    return {counter.key.metric.name: counter.committed for counter in sorted(
        counters, key=lambda c: c.key.metric.name)}

Run it on the lab's data. The result is a sorted CSV file, and the counters of the run.

In [11]:
counters = run_wordcount('data/*', 'outputs/word_counts', top_n=20)
print('counters:', counters)
print(''.join(open('outputs/word_counts.csv').readlines()[:6]))

counters: {'lines': 5525, 'lines_without_words': 1665, 'words': 27986}
word,count
the,910
and,738
i,652
to,580
of,483



A short stopword list, written once. It holds 25 very common words that were chosen for this lab. It is not a standard list.

In [12]:
STOPWORDS = 'the and i to of a you my that in is not it me with for his he be your this but have as him'.split()
if not os.path.exists('stopwords.txt'):
    with open('stopwords.txt', 'w') as handle:
        handle.write('\n'.join(STOPWORDS) + '\n')

run_wordcount('data/*', 'outputs/word_counts_no_stopwords', top_n=8, stopwords_path='stopwords.txt')
print(open('outputs/word_counts_no_stopwords.csv').read())

word,count
king,311
lear,253
thou,220
kent,175
gloucester,167
thy,161
no,154
so,141



The input check stops early, with a clear message, and writes nothing:

In [13]:
try:
    run_wordcount('data/nothing*.txt')
except ValueError as error:
    print('stopped:', error)

stopped: no file matches 'data/nothing*.txt'


## 3. A second dataset

The lab reads `kinglear.txt` from the public bucket `gs://dataflow-samples/shakespeare/`. The same bucket holds the other plays.
This cell downloads four tragedies and four comedies over HTTPS and checks each file against a SHA-256 checksum, so the data is the same on every machine.
A file that is already there with the right checksum is not downloaded again. The grouping into genres is a choice made for this lab.

In [14]:
PLAY_SHA256 = {
    'comedies/asyoulikeit.txt': 'eaa3526fe53859f34ecdf255712f9ecf0b2c903451d4755b2edaa2e2599cb0fc',
    'comedies/midsummersnightsdream.txt': '684b22072f158a7965e942d7e9b633bf7c6c65b14a7b519631bbf2f826349370',
    'comedies/muchadoaboutnothing.txt': 'fad832e284ccabb6fcb5dc3dd34d2636a07358d6474f10eed8e9ed98050208ee',
    'comedies/twelfthnight.txt': '68a44a35e82462d4f0217d2a40db874a431d666626691a806b63339118831039',
    'tragedies/hamlet.txt': '1846810b98f3dce4174a488cd19f3e7ceabff2a4b00487089fbcb2891003d847',
    'tragedies/kinglear.txt': '674a2725884307c96398440497c889ad8cecccedf5689df85e6b0faabe4e0fe8',
    'tragedies/macbeth.txt': '339afa674ac54e5038bd610056e87b0ba70f6543b2fbcfea421bfd0fbbc21e7f',
    'tragedies/othello.txt': 'e1590d00e121e53587157fa952e0da4070b76721599ebcedce3bb698228adbab',
}

BUCKET = 'https://storage.googleapis.com/dataflow-samples/shakespeare/'

In [15]:
import hashlib
import urllib.request
from pathlib import Path


def fetch_plays():
    """Download the eight plays into plays/<genre>/<play>.txt, and verify each checksum."""
    for relative, expected in PLAY_SHA256.items():
        path = Path('plays') / relative
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected:
            continue
        path.parent.mkdir(parents=True, exist_ok=True)
        data = urllib.request.urlopen(BUCKET + path.name).read()
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError('checksum mismatch for ' + relative)
        path.write_bytes(data)
    return len(PLAY_SHA256)


print(fetch_plays(), 'plays verified')

8 plays verified


## 4. Comparing the plays and the genres

This is a second pipeline. It reads every play under `plays/<genre>/<play>.txt` and writes three CSV files:

| File | What it holds |
|---|---|
| `play_stats.csv` | Words, different words, and different words per word said, for each play |
| `play_top_words.csv` | The ten most common words of each play, without stopwords |
| `genre_words.csv` | The words that one genre uses most compared with the other |

The Beam ideas that it adds to the lab's word count:

1. `ReadFromTextWithFilename` gives each line together with its file name. The name gives the genre and the play.
2. `Count.PerElement` counts each `(genre, play, word)`.
3. `PlayStats` is a custom combiner. It folds the counts of one play into two numbers.
4. The total number of words in each genre reaches every word through a side input.
5. `WriteCsv` is a composite transform: it gathers rows, sorts them and writes one CSV file.

A rate is how many times a word appears in every 10,000 words of a genre. A word is ranked only if it appears in at least 3 plays and at least 30 times.
Words are ranked by the ratio of the two rates, with 0.5 added to each count so a word that one genre never says does not divide by zero.

In [16]:
def play_key(path):
    """Return (genre, play) from a path such as plays/tragedies/hamlet.txt."""
    return os.path.basename(os.path.dirname(path)), os.path.splitext(os.path.basename(path))[0]


def find_genres(inputs_pattern):
    """Return the sorted genre folder names that the pattern matches. A comparison needs exactly two."""
    genres = sorted({play_key(path)[0] for path in glob.glob(inputs_pattern)})
    if len(genres) != 2:
        raise ValueError("need files in exactly 2 genre folders, found {}: {}".format(len(genres), genres))
    return tuple(genres)


def words_of(path_and_line):
    """Turn one (path, line) element into one (genre, play, word) element per word."""
    path, line = path_and_line
    genre, play = play_key(path)
    for word in find_words(line):
        yield genre, play, word

In [17]:
class PlayStats(beam.CombineFn):
    """Fold the per-word counts of one play into (words, distinct words)."""

    def create_accumulator(self):
        return 0, 0

    def add_input(self, accumulator, count):
        return accumulator[0] + count, accumulator[1] + 1

    def merge_accumulators(self, accumulators):
        words = distinct = 0
        for part_words, part_distinct in accumulators:
            words += part_words
            distinct += part_distinct
        return words, distinct

    def extract_output(self, accumulator):
        return accumulator


def play_stat_row(key, stats):
    """One CSV row per play: genre, play, words, distinct words, distinct words per word said."""
    (genre, play), (words, distinct) = key, stats
    return genre, play, words, distinct, round(distinct / words, 4)


def add_pairs(pairs):
    """Add up (count, plays) pairs."""
    pairs = list(pairs)
    return sum(count for count, _ in pairs), sum(plays for _, plays in pairs)

In [18]:
def genre_rows(word, entries, totals, genres, min_plays, min_count):
    """Return a list with one row for this word, or an empty list if it is too rare to trust or one letter long.

    entries: (genre, count, plays) for each genre that uses the word. totals: {genre: words in genre}.
    The rate is per 10,000 words, with 0.5 added to each count so a missing word does not divide by zero.
    """
    if len(word) == 1:  # a one-letter token such as t, from the elision 't, is not a word to compare
        return []
    first, second = genres
    seen = {genre: (count, plays) for genre, count, plays in entries}
    count_a, plays_a = seen.get(first, (0, 0))
    count_b, plays_b = seen.get(second, (0, 0))
    if plays_a + plays_b < min_plays or count_a + count_b < min_count:
        return []
    rate_a = (count_a + 0.5) / totals[first] * 10000
    rate_b = (count_b + 0.5) / totals[second] * 10000
    ratio = rate_b / rate_a
    favours = second if ratio > 1 else first
    return [(word, favours, count_a, count_b, round(rate_a, 2), round(rate_b, 2), round(ratio, 3),
             plays_a + plays_b)]


def score_word(word_and_entries, totals, genres, min_plays, min_count):
    """Unpack one (word, entries) element for genre_rows. A side input needs a plain FlatMap."""
    word, entries = word_and_entries
    return genre_rows(word, entries, totals, genres, min_plays, min_count)


def pick_extremes(rows, top_n):
    """Keep the top_n words that favour the second genre (highest ratio) and the top_n that favour the first."""
    by_ratio = sorted(rows, key=lambda row: (-row[6], row[0]))
    favour_second = [row for row in by_ratio if row[6] > 1][:top_n]
    favour_first = sorted((row for row in rows if row[6] <= 1), key=lambda row: (row[6], row[0]))[:top_n]
    return favour_second + favour_first

In [19]:
def rows_to_csv(rows):
    """Make the CSV body, one line per row."""
    return '\n'.join(','.join(str(value) for value in row) for row in rows)


class WriteCsv(beam.PTransform):
    """Gather rows into one list, arrange it (sort or cut), and write one CSV file with a header."""

    def __init__(self, prefix, header, arrange):
        super().__init__()
        self.prefix, self.header, self.arrange = prefix, header, arrange

    def expand(self, rows):
        return (
            rows
            | 'Gather rows' >> beam.combiners.ToList()  # one list of tuples
            | 'Arrange rows' >> beam.Map(self.arrange)  # one arranged list
            | 'Make CSV body' >> beam.Map(rows_to_csv)  # one str, so the lines keep their order
            | 'Write file' >> beam.io.WriteToText(
                self.prefix, file_name_suffix='.csv', shard_name_template='', header=self.header)
        )

In [20]:
def build_compare_pipeline(pipeline, inputs_pattern, output_dir, genres, stopwords=frozenset(),
                   top_n=10, genre_top_n=15, min_plays=3, min_count=30):
    """Attach all the comparison steps to a pipeline."""
    first, second = genres
    counts = (
        pipeline
        | 'Read lines' >> beam.io.textio.ReadFromTextWithFilename(inputs_pattern)  # (path, line)
        | 'Find words' >> beam.FlatMap(words_of)  # (genre, play, word)
        | 'Count per play' >> beam.combiners.Count.PerElement()  # ((genre, play, word), count)
    )

    # 1. Statistics per play: a custom combiner folds each play's word counts into two numbers.
    (
        counts
        | 'Key by play' >> beam.MapTuple(lambda key, count: ((key[0], key[1]), count))
        | 'Play stats' >> beam.CombinePerKey(PlayStats())  # ((genre, play), (words, distinct))
        | 'Play stat rows' >> beam.MapTuple(play_stat_row)
        | 'Write play stats' >> WriteCsv(
            os.path.join(output_dir, 'play_stats'), 'genre,play,words,distinct_words,distinct_per_word',
            lambda rows: sorted(rows))
    )

    # 2. The top words of each play, without stopwords.
    (
        counts
        | 'Keep real words' >> beam.Filter(lambda element: element[0][2] not in stopwords)
        | 'Key by play, value word' >> beam.MapTuple(lambda key, count: ((key[0], key[1]), (key[2], count)))
        | 'Group words per play' >> beam.GroupByKey()
        | 'Rank per play' >> beam.FlatMapTuple(
            lambda key, pairs: [(key[0], key[1], place, word, count)
                                for place, (word, count) in enumerate(rank(pairs, top_n), 1)])
        | 'Write top words' >> WriteCsv(
            os.path.join(output_dir, 'play_top_words'), 'genre,play,rank,word,count',
            lambda rows: sorted(rows))
    )

    # 3. Words that one genre uses much more than the other. The genre totals reach every word
    #    through a side input.
    genre_word = (
        counts
        | 'Key by genre and word' >> beam.MapTuple(lambda key, count: ((key[0], key[2]), (count, 1)))
        | 'Add counts and plays' >> beam.CombinePerKey(add_pairs)  # ((genre, word), (count, plays))
    )
    totals = (
        genre_word
        | 'Key by genre' >> beam.MapTuple(lambda key, pair: (key[0], pair[0]))
        | 'Genre totals' >> beam.CombinePerKey(sum)  # (genre, words in genre)
    )
    (
        genre_word
        | 'Key by word' >> beam.MapTuple(lambda key, pair: (key[1], (key[0], pair[0], pair[1])))
        | 'Group genres per word' >> beam.GroupByKey()
        | 'Score words' >> beam.FlatMap(
            score_word, beam.pvalue.AsDict(totals), genres, min_plays, min_count)
        | 'Write genre words' >> WriteCsv(
            os.path.join(output_dir, 'genre_words'),
            'word,favours,{0}_count,{1}_count,{0}_per_10k,{1}_per_10k,ratio_{1}_over_{0},plays'.format(
                first, second),
            lambda rows: pick_extremes(rows, genre_top_n))
    )

In [21]:
def run_compare(inputs_pattern='plays/*/*.txt', output_dir='outputs', stopwords_path='stopwords.txt',
                top_n=10, genre_top_n=15, min_plays=3, min_count=30):
    """Run the comparison and write the three CSV files. Raises ValueError (or OSError) on bad input."""
    if min(top_n, genre_top_n, min_plays, min_count) < 1:
        raise ValueError('top_n, genre_top_n, min_plays and min_count must be 1 or more')
    check_input(inputs_pattern)
    genres = find_genres(inputs_pattern)
    stopwords = load_stopwords(stopwords_path)
    os.makedirs(output_dir, exist_ok=True)
    with beam.Pipeline() as pipeline:
        build_compare_pipeline(pipeline, inputs_pattern, output_dir, genres, stopwords,
                               top_n, genre_top_n, min_plays, min_count)

Run it. The three files go to `outputs/`.

In [22]:
run_compare('plays/*/*.txt', 'outputs')
for name in ('play_stats', 'genre_words'):
    print(name + '.csv')
    print(open('outputs/' + name + '.csv').read())

play_stats.csv
genre,play,words,distinct_words,distinct_per_word
comedies,asyoulikeit,23054,3272,0.1419
comedies,midsummersnightsdream,17332,3005,0.1734
comedies,muchadoaboutnothing,22718,2979,0.1311
comedies,twelfthnight,21570,3118,0.1446
tragedies,hamlet,32396,4728,0.1459
tragedies,kinglear,27986,4176,0.1492
tragedies,macbeth,18414,3358,0.1824
tragedies,othello,28059,3801,0.1355

genre_words.csv
word,favours,comedies_count,tragedies_count,comedies_per_10k,tragedies_per_10k,ratio_tragedies_over_comedies,plays
castle,tragedies,0,47,0.06,4.45,75.28,4
ghost,tragedies,0,39,0.06,3.7,62.601,3
king,tragedies,11,545,1.36,51.05,37.588,8
witch,tragedies,1,57,0.18,5.38,30.376,4
doctor,tragedies,1,46,0.18,4.35,24.565,4
france,tragedies,1,37,0.18,3.51,19.81,3
daughters,tragedies,1,32,0.18,3.04,17.169,5
majesty,tragedies,1,29,0.18,2.76,15.584,4
soldiers,tragedies,1,29,0.18,2.76,15.584,5
general,tragedies,2,47,0.3,4.45,15.056,5
england,tragedies,2,32,0.3,3.04,10.301,5
murder,tragedies,3,43,0.41,4.07

## 5. A dashboard

`build_report` turns the three CSV files into one HTML page: bars for the length of each play, two charts of the words each genre favours, and a table of the top words.
The page uses only inline SVG, so it needs no network and no JavaScript. It has a light and a dark mode, a tooltip on every bar, and a table view under each chart.
Comedies are always blue and tragedies are always orange. The two colours were checked for colour-blind readers and for contrast on the light and the dark background.

In [23]:
import csv
import html
import math

In [24]:
WIDTH, LEFT, RIGHT = 720, 204, 60
PLOT = WIDTH - LEFT - RIGHT

In [25]:
STYLE = """
:root {
  color-scheme: light dark;
  --surface: #fcfcfb; --text: #0b0b0b; --text-2: #52514e; --grid: #e6e5e1;
  --series-1: #2a78d6; --series-2: #eb6834;
}
@media (prefers-color-scheme: dark) {
  :root {
    --surface: #1a1a19; --text: #f0efec; --text-2: #c3c2b7; --grid: #2f2f2d;
    --series-1: #3987e5; --series-2: #d95926;
  }
}
html, body { margin: 0; background: var(--surface); }
.viz-root { max-width: 760px; margin: 0 auto; padding: 24px 20px 48px; color: var(--text);
  font: 15px/1.5 -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
h1 { font-size: 26px; margin: 0 0 4px; } h2 { font-size: 18px; margin: 36px 0 4px; }
p { margin: 4px 0 10px; } .sub { color: var(--text-2); }
.legend { display: flex; gap: 18px; margin: 8px 0 4px; color: var(--text-2); font-size: 13px; }
.legend span::before { content: ""; display: inline-block; width: 10px; height: 10px; margin-right: 6px;
  border-radius: 2px; background: var(--c); }
svg { width: 100%; height: auto; display: block; }
svg text { font: 12px -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; fill: var(--text-2); }
svg text.value { fill: var(--text); } svg text.word { fill: var(--text); font-size: 13px; }
svg .grid { stroke: var(--grid); stroke-width: 1; }
.s1 { fill: var(--series-1); } .s2 { fill: var(--series-2); }
.s1.key { --c: var(--series-1); } .s2.key { --c: var(--series-2); }
table { border-collapse: collapse; width: 100%; font-size: 13px; margin-top: 8px; }
th, td { text-align: left; padding: 4px 10px 4px 0; border-bottom: 1px solid var(--grid); }
th { color: var(--text-2); font-weight: 600; } td.num, th.num { text-align: right; font-variant-numeric: tabular-nums; }
details { margin-top: 6px; } summary { cursor: pointer; color: var(--text-2); font-size: 13px; }
.note { color: var(--text-2); font-size: 13px; }
"""

In [26]:
TITLES = {
    'asyoulikeit': 'As You Like It', 'hamlet': 'Hamlet', 'kinglear': 'King Lear', 'macbeth': 'Macbeth',
    'midsummersnightsdream': "A Midsummer Night's Dream", 'muchadoaboutnothing': 'Much Ado About Nothing',
    'othello': 'Othello', 'twelfthnight': 'Twelfth Night',
}


def title_of(play):
    """The play's title, or the file name if the play is not one of the eight known ones."""
    return TITLES.get(play, play)


def read_rows(path):
    with open(path, newline='') as handle:
        return list(csv.DictReader(handle))

In [27]:
def nice_ticks(maximum):
    """Return (axis maximum, tick values): a clean step of 1, 2 or 5 times a power of ten, 5 ticks at most."""
    step = 10 ** math.floor(math.log10(maximum / 4))
    for factor in (1, 2, 5, 10):
        if maximum / (step * factor) <= 5:
            step *= factor
            break
    top = math.ceil(maximum / step) * step
    return top, [i * step for i in range(int(round(top / step)) + 1)]


def tick_label(value):
    return '{:g}'.format(value) if value < 1000 else '{:.0f}K'.format(value / 1000)


def bar_path(x, y, width, height, radius=4):
    """A bar that is square at its baseline (left) and rounded at its data end (right)."""
    radius = min(radius, width / 2, height / 2)
    return ('M{x:.1f},{y:.1f} H{xr:.1f} A{r:.1f},{r:.1f} 0 0 1 {x2:.1f},{yr:.1f} V{yb:.1f} '
            'A{r:.1f},{r:.1f} 0 0 1 {xr:.1f},{y2:.1f} H{x:.1f} Z').format(
        x=x, y=y, xr=x + width - radius, r=radius, x2=x + width, yr=y + radius, yb=y + height - radius,
        y2=y + height)

In [28]:
def axis(top, ticks, plot_height, label):
    """Hairline vertical gridlines with tick labels below, and an axis title."""
    parts = []
    for tick in ticks:
        x = LEFT + tick / top * PLOT
        parts.append('<line class="grid" x1="{0:.1f}" x2="{0:.1f}" y1="0" y2="{1}"/>'.format(x, plot_height))
        parts.append('<text x="{0:.1f}" y="{1}" text-anchor="middle">{2}</text>'.format(
            x, plot_height + 16, tick_label(tick)))
    parts.append('<text x="{0}" y="{1}" text-anchor="end">{2}</text>'.format(
        LEFT + PLOT, plot_height + 34, html.escape(label)))
    return ''.join(parts)


def svg(height, body, label):
    return '<svg viewBox="0 0 {} {}" role="img" aria-label="{}">{}</svg>'.format(
        WIDTH, height, html.escape(label), body)


def legend(genres):
    return '<div class="legend">{}</div>'.format(''.join(
        '<span class="s{} key">{}</span>'.format(i + 1, html.escape(genre)) for i, genre in enumerate(genres)))

In [29]:
def play_chart(plays, genres):
    """One bar per play: its number of words, coloured by genre."""
    ordered = sorted(plays, key=lambda row: -int(row['words']))
    pitch, bar = 30, 18
    top, ticks = nice_ticks(max(int(row['words']) for row in ordered))
    height = pitch * len(ordered)
    parts = [axis(top, ticks, height, 'words in the play')]
    for i, row in enumerate(ordered):
        y = i * pitch + (pitch - bar) / 2
        words = int(row['words'])
        series = genres.index(row['genre']) + 1
        parts.append('<g><title>{}: {:,} words, {:,} different words ({})</title>'.format(
            html.escape(title_of(row['play'])), words, int(row['distinct_words']), html.escape(row['genre'])))
        parts.append('<text class="word" x="{}" y="{:.1f}" text-anchor="end">{}</text>'.format(
            LEFT - 8, y + bar / 2 + 4, html.escape(title_of(row['play']))))
        width = words / top * PLOT
        parts.append('<path class="s{}" d="{}"/>'.format(series, bar_path(LEFT, y, width, bar)))
        parts.append('<text class="value" x="{:.1f}" y="{:.1f}">{:,}</text></g>'.format(
            LEFT + width + 6, y + bar / 2 + 4, words))
    return svg(height + 44, ''.join(parts), 'Words per play, one bar per play, coloured by genre')


def genre_chart(rows, genres, label):
    """Per word, two thin bars (one per genre): how often each genre says it per 10,000 words."""
    first, second = genres
    bar, gap, pitch = 9, 2, 30
    top, ticks = nice_ticks(max(max(float(r[first + '_per_10k']), float(r[second + '_per_10k'])) for r in rows))
    height = pitch * len(rows)
    parts = [axis(top, ticks, height, 'times per 10,000 words')]
    for i, row in enumerate(rows):
        y0 = i * pitch + (pitch - (2 * bar + gap)) / 2
        rates = [float(row[first + '_per_10k']), float(row[second + '_per_10k'])]
        counts = [int(row[first + '_count']), int(row[second + '_count'])]
        parts.append('<g><title>{}: {:g} per 10,000 words in {} ({} times); {:g} in {} ({} times)</title>'.format(
            html.escape(row['word']), rates[0], html.escape(first), counts[0], rates[1], html.escape(second),
            counts[1]))
        parts.append('<text class="word" x="{}" y="{:.1f}" text-anchor="end">{}</text>'.format(
            LEFT - 8, y0 + bar + 2, html.escape(row['word'])))
        for k, rate in enumerate(rates):
            y = y0 + k * (bar + gap)
            width = max(rate / top * PLOT, 1.5)
            parts.append('<path class="s{}" d="{}"/>'.format(k + 1, bar_path(LEFT, y, width, bar, 3)))
        larger = 0 if rates[0] >= rates[1] else 1  # label only the favoured side, so the chart stays calm
        parts.append('<text class="value" x="{:.1f}" y="{:.1f}">{:.2f}</text></g>'.format(
            LEFT + max(rates[larger] / top * PLOT, 1.5) + 6, y0 + larger * (bar + gap) + bar - 1, rates[larger]))
    return svg(height + 44, ''.join(parts), label)

In [30]:
def table(headers, rows, numeric=()):
    head = ''.join('<th{}>{}</th>'.format(' class="num"' if i in numeric else '', html.escape(h))
                   for i, h in enumerate(headers))
    body = ''.join('<tr>{}</tr>'.format(''.join(
        '<td{}>{}</td>'.format(' class="num"' if i in numeric else '', html.escape(str(cell)))
        for i, cell in enumerate(row))) for row in rows)
    return '<table><thead><tr>{}</tr></thead><tbody>{}</tbody></table>'.format(head, body)


def render(play_stats, top_words, genre_words):
    genres = sorted({row['genre'] for row in play_stats})
    if len(genres) != 2:
        raise ValueError('the report needs exactly 2 genres, found {}'.format(genres))
    first, second = genres
    total_words = sum(int(row['words']) for row in play_stats)

    plays_table = table(
        ['play', 'genre', 'words', 'different words', 'different words per word said'],
        [(title_of(r['play']), r['genre'], '{:,}'.format(int(r['words'])), '{:,}'.format(int(r['distinct_words'])),
          r['distinct_per_word']) for r in sorted(play_stats, key=lambda r: -int(r['words']))],
        numeric=(2, 3, 4))

    def genre_section(favoured, title, intro):
        rows = [r for r in genre_words if r['favours'] == favoured]
        if not rows:
            return '<h2>{}</h2><p class="sub">No word passed the filters described below.</p>'.format(
                html.escape(title))
        words_table = table(
            ['word', first + ' times', second + ' times', first + ' per 10,000', second + ' per 10,000',
             'plays that use it'],
            [(r['word'], r[first + '_count'], r[second + '_count'], r[first + '_per_10k'],
              r[second + '_per_10k'], r['plays']) for r in rows], numeric=(1, 2, 3, 4, 5))
        chart = genre_chart(rows, genres, '{}: bars show how often {} and {} say each word'.format(
            title, first, second))
        return '<h2>{}</h2><p class="sub">{}</p>{}{}<details><summary>Table view</summary>{}</details>'.format(
            html.escape(title), intro, legend(genres), chart, words_table)

    top_rows = []
    for play in sorted({(r['genre'], r['play']) for r in top_words}):
        words = [r for r in top_words if (r['genre'], r['play']) == play and int(r['rank']) <= 5]
        top_rows.append((title_of(play[1]), play[0], ', '.join('{} ({})'.format(w['word'], w['count']) for w in words)))

    return '''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Word use in {n} Shakespeare plays</title><style>{style}</style></head>
<body><main class="viz-root">
<h1>Word use in {n} Shakespeare plays</h1>
<p class="sub">{words:,} words in {n} plays, counted by an Apache Beam pipeline: {a} versus {b}.</p>

<h2>How long is each play?</h2>
<p class="sub">Bars show the number of words. Colour shows the genre.</p>
{legend}{play_chart}
<details><summary>Table view</summary>{plays_table}</details>

{tragic}
{comic}

<h2>The top words in each play</h2>
<p class="sub">The five most common words, without the very common ones listed in <code>stopwords.txt</code>.</p>
{top_table}

<h2>How to read this</h2>
<p class="note">A rate is how many times a word appears in every 10,000 words of a genre. A word is shown only
if it appears in at least 3 plays and at least 30 times, so a name from one play cannot win. Words are ranked
by the ratio of the two rates, with 0.5 added to each count so a word that one genre never says does not divide
by zero. The grouping of plays into genres is a choice made for this lab. Character names can still pass the
filter when the same word is also used elsewhere: <em>bottom</em> is a character in one comedy and a plain word
in three other plays.</p>
</main></body></html>
'''.format(
        n=len(play_stats), style=STYLE, words=total_words, a=html.escape(first), b=html.escape(second),
        legend=legend(genres), play_chart=play_chart(play_stats, genres), plays_table=plays_table,
        tragic=genre_section(second, 'Words ' + second + ' use more', 'Each word is more common in ' + second + '.'),
        comic=genre_section(first, 'Words ' + first + ' use more', 'Each word is more common in ' + first + '.'),
        top_table=table(['play', 'genre', 'top words'], top_rows))

In [31]:
def build_report(output_dir='outputs'):
    """Read the three CSV files of run_compare, write output_dir/report.html and return the page."""
    try:
        files = [read_rows(os.path.join(output_dir, name + '.csv'))
                 for name in ('play_stats', 'play_top_words', 'genre_words')]
    except OSError as error:
        raise ValueError('{} (run the comparison first)'.format(error)) from error
    page = render(*files)
    with open(os.path.join(output_dir, 'report.html'), 'w') as handle:
        handle.write(page)
    return page

Build the page and show it. The page is shown inside a frame, so its colours do not change the rest of the notebook.

In [32]:
import warnings
from IPython.display import HTML, display

# IPython suggests IFrame, which needs a file address. This frame holds the page itself.
warnings.filterwarnings('ignore', message='Consider using IPython.display.IFrame')
page = build_report('outputs')
display(HTML('<iframe srcdoc="{}" width="100%" height="2500" style="border:0"></iframe>'.format(
    html.escape(page, quote=True))))

A saved picture of the same page, for viewers that do not show frames (GitHub, for example):

![The report: bars for the length of each play, two charts of genre words, and a table of top words](docs/report.png)

## 6. Who speaks, and how long are the words?

Two more questions about the same eight plays, each answered by its own pipeline.

**Who speaks the most in each play?** A play file is laid out as `NAME<tab>words` for the first line of a speech, with the next lines of the same speech indented by a tab.
Stage directions are in `[brackets]`, and each scene starts with an `ACT` line and a `SCENE` line. The cast list before the first `ACT I` has the same `NAME<tab>words` shape, so it must be skipped.
A speech runs over many lines, so one line alone is not enough. This pipeline reads each whole play as one element with `fileio.MatchFiles` and `fileio.ReadMatches`, and splits it into speeches.

**How long are the words in each genre?** Every word is measured, and each length is counted as a percent of the genre's words. Words of 15 letters or more are pooled.

In [33]:
from apache_beam.io import fileio

BRACKETS = re.compile(r'\[[^\]]*\]')  # a stage direction such as [Aside], which may run over several lines
HEADING = re.compile(r'^(ACT|SCENE)\b')  # ACT I, SCENE II, and so on


def count_spoken(text):
    """The number of words in some spoken text, without the stage directions in [brackets]."""
    return len(find_words(BRACKETS.sub(' ', text)))


def parse_speeches(text):
    """Return one (speaker, words) pair for each speech in a whole play, in order.

    A speech starts with NAME<tab>words. The lines after it that start with a space or a tab continue
    the speech. A line that starts with ACT or SCENE ends it. The cast list before the first 'ACT I'
    line has the same NAME<tab>words shape, so it is skipped. A speaker's name is written the same
    way everywhere: no colon at the end, and title case.
    """
    speeches, current, started = [], None, False
    for line in text.splitlines():
        if not started:
            started = line.strip() == 'ACT I'
            continue
        if not line.strip():
            continue
        if line[0] in ' \t':
            if current is not None:
                current[1].append(line)
        elif HEADING.match(line):
            current = None
        elif '\t' in line:
            speaker, spoken = line.split('\t', 1)
            current = [speaker.strip().rstrip(':').strip().title(), [spoken]]
            speeches.append(current)
        else:
            current = None
    return [(speaker, count_spoken('\n'.join(parts))) for speaker, parts in speeches]

In [34]:
def read_whole_play(readable_file):
    """One (path, text) element for a whole file. A speech runs over many lines, so one line is not enough."""
    return readable_file.metadata.path, readable_file.read_utf8()


def speech_elements(path_and_text):
    """Turn one play into one ((genre, play, speaker), (1, words)) element for each speech."""
    path, text = path_and_text
    genre, play = play_key(path)
    for speaker, words in parse_speeches(text):
        yield (genre, play, speaker), (1, words)


def sum_pairs(pairs):
    """Add up (speeches, words) pairs."""
    pairs = list(pairs)
    return sum(speeches for speeches, _ in pairs), sum(words for _, words in pairs)


def top_speakers(play, speakers, top_n):
    """Rank the (speaker, speeches, words) triples of one play, most speeches first, and keep top_n."""
    genre, name = play
    ranked = sorted(speakers, key=lambda triple: (-triple[1], -triple[2], triple[0]))[:top_n]
    return [(genre, name, place, speaker, speeches, words)
            for place, (speaker, speeches, words) in enumerate(ranked, 1)]


def build_speakers_pipeline(pipeline, inputs_pattern, output_dir, top_n=5):
    """Attach the speaker steps to a pipeline. Each play is read as one element."""
    (
        pipeline
        | 'Match files' >> fileio.MatchFiles(inputs_pattern)
        | 'Open files' >> fileio.ReadMatches()
        | 'Read whole plays' >> beam.Map(read_whole_play)  # (path, text)
        | 'Find speeches' >> beam.FlatMap(speech_elements)  # ((genre, play, speaker), (1, words))
        | 'Add up per speaker' >> beam.CombinePerKey(sum_pairs)  # ((genre, play, speaker), (speeches, words))
        | 'Key by play' >> beam.MapTuple(lambda key, total: ((key[0], key[1]), (key[2], total[0], total[1])))
        | 'Group speakers per play' >> beam.GroupByKey()
        | 'Rank speakers' >> beam.FlatMapTuple(lambda play, speakers: top_speakers(play, speakers, top_n))
        | 'Write speakers' >> WriteCsv(
            os.path.join(output_dir, 'speakers'), 'genre,play,rank,speaker,speeches,words',
            lambda rows: sorted(rows))
    )


def run_speakers(inputs_pattern='plays/*/*.txt', output_dir='outputs', top_n=5):
    """Write output_dir/speakers.csv: the top_n speakers of each play. Raises ValueError on bad input."""
    if top_n < 1:
        raise ValueError('top_n must be 1 or more')
    check_input(inputs_pattern)
    os.makedirs(output_dir, exist_ok=True)
    with beam.Pipeline() as pipeline:
        build_speakers_pipeline(pipeline, inputs_pattern, output_dir, top_n)

In [35]:
def length_row(counted, totals):
    """One CSV row: genre, word length, number of words of that length, and their percent of the genre."""
    (genre, length), count = counted
    return genre, length, count, round(count / totals[genre] * 100, 2)


def build_word_lengths_pipeline(pipeline, inputs_pattern, output_dir, longest=15):
    """Attach the word-length steps to a pipeline. Words of `longest` letters or more are pooled."""
    counts = (
        pipeline
        | 'Read lines' >> beam.io.textio.ReadFromTextWithFilename(inputs_pattern)  # (path, line)
        | 'Find words' >> beam.FlatMap(words_of)  # (genre, play, word)
        | 'Length of each word' >> beam.Map(lambda element: (element[0], min(len(element[2]), longest)))
        | 'Count lengths' >> beam.combiners.Count.PerElement()  # ((genre, length), count)
    )
    totals = (
        counts
        | 'Key by genre' >> beam.MapTuple(lambda key, count: (key[0], count))
        | 'Words per genre' >> beam.CombinePerKey(sum)  # (genre, words in genre)
    )
    (
        counts
        | 'Percent of the genre' >> beam.Map(length_row, beam.pvalue.AsDict(totals))
        | 'Write word lengths' >> WriteCsv(
            os.path.join(output_dir, 'word_lengths'), 'genre,length,words,percent', lambda rows: sorted(rows))
    )


def run_word_lengths(inputs_pattern='plays/*/*.txt', output_dir='outputs', longest=15):
    """Write output_dir/word_lengths.csv: how many words of each length each genre uses."""
    if longest < 1:
        raise ValueError('longest must be 1 or more')
    check_input(inputs_pattern)
    os.makedirs(output_dir, exist_ok=True)
    with beam.Pipeline() as pipeline:
        build_word_lengths_pipeline(pipeline, inputs_pattern, output_dir, longest)

The top three speakers of each play, by number of speeches:

In [36]:
run_speakers('plays/*/*.txt', 'outputs', top_n=5)
rows = [line.split(',') for line in open('outputs/speakers.csv').read().splitlines()[1:]]
print('{:<10} {:<26} {:<16} {:>8} {:>7}'.format('genre', 'play', 'speaker', 'speeches', 'words'))
for genre, play, rank, speaker, speeches, words in rows:
    if int(rank) <= 3:
        print('{:<10} {:<26} {:<16} {:>8} {:>7}'.format(genre, TITLES.get(play, play), speaker, speeches, words))

genre      play                       speaker          speeches   words
comedies   As You Like It             Rosalind              201    5737
comedies   As You Like It             Orlando               120    2398
comedies   As You Like It             Celia                 108    2119
comedies   A Midsummer Night's Dream  Bottom                 50    1691
comedies   A Midsummer Night's Dream  Lysander               50    1400
comedies   A Midsummer Night's Dream  Theseus                48    1734
comedies   Much Ado About Nothing     Don Pedro             135    2485
comedies   Much Ado About Nothing     Benedick              134    3766
comedies   Much Ado About Nothing     Claudio               125    2075
comedies   Twelfth Night              Sir Toby Belch        152    2705
comedies   Twelfth Night              Viola                 121    2573
comedies   Twelfth Night              Olivia                118    2320
tragedies  Hamlet                     Hamlet                359 

The share of each word length, side by side for the two genres:

In [37]:
run_word_lengths('plays/*/*.txt', 'outputs', longest=15)
rows = [line.split(',') for line in open('outputs/word_lengths.csv').read().splitlines()[1:]]
percent = {(genre, int(length)): float(share) for genre, length, count, share in rows}
print('{:>7} {:>10} {:>10}'.format('length', 'comedies', 'tragedies'))
for length in range(1, 16):
    print('{:>7} {:>9.2f}% {:>9.2f}%'.format(str(length) if length < 15 else '15+', percent.get(('comedies', length), 0),
                                           percent.get(('tragedies', length), 0)))

 length   comedies  tragedies
      1      5.16%      4.29%
      2     16.96%     16.33%
      3     21.93%     20.75%
      4     22.88%     23.49%
      5     12.07%     12.06%
      6      7.91%      8.85%
      7      5.88%      6.50%
      8      3.88%      3.74%
      9      1.89%      2.13%
     10      0.85%      1.08%
     11      0.39%      0.48%
     12      0.14%      0.23%
     13      0.04%      0.05%
     14      0.01%      0.01%
    15+      0.01%      0.01%
